# Mask and Bounding Box Overlay Generation
This notebook generates image overlays. It intelligently loads an original image corresponding to a mask from `polyp_masks`, `instruments_masks`, `pseudo_masks`, or `gradcam_masks`.
To preserve visual textures and colors, this tool draws the **contour outline** of the mask in green and a **bounding box** in red.

In [ ]:
import cv2
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import csv
import json

DATA_DIR = '../data'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')

### Overlay Function

In [4]:
def get_original_image_path(mask_path):
    """Extracts the exact image ID from the mask's filename or parent directory and returns the path to the original image."""
    p = Path(mask_path)
    img_id = None
    
    normalized_path = str(p.as_posix())
    
    if 'polyp_masks' in normalized_path or 'instruments_masks' in normalized_path:
        img_id = p.stem
    elif 'pseudo_masks' in normalized_path:
        img_id = p.name.split('_')[0]
    elif 'gradcam_masks' in normalized_path:
        img_id = p.parent.name
        
    if img_id:
        img_path = os.path.join(IMAGES_DIR, f"{img_id}.jpg")
        if os.path.exists(img_path):
            return img_path
    return None

def load_valid_instrument_ids():
    valid_ids = set()
    csv_path = 'data/combined/instruments_mask_phrases_v2.csv'
    if os.path.exists(csv_path):
        with open(csv_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                valid_ids.add(row['img_id'])
    return valid_ids

VALID_INSTRUMENT_IDS = load_valid_instrument_ids()

def should_process_mask(mask_path):
    p = Path(mask_path)
    normalized_path = str(p.as_posix())
    
    if 'instruments_masks' in normalized_path:
        img_id = p.stem
        if img_id not in VALID_INSTRUMENT_IDS:
            return False
            
    elif 'pseudo_masks' in normalized_path:
        if 'cecum' not in p.name and 'z-line' not in p.name:
            return False
            
    elif 'gradcam_masks' in normalized_path:
        class_name = p.parent.parent.name
        if class_name not in ['ulcerative_colitis', 'oesophagitis']:
            return False
        
        bbox_json_path = p.parent / 'bbox_data.json'
        if bbox_json_path.exists():
            with open(bbox_json_path, 'r') as f:
                bbox_data = json.load(f)
                if bbox_data.get('prediction', 0) <= 0.80:
                    return False
        else:
            return False
            
    return True

def process_mask_and_image(mask_path, output_path=None, show=True):
    if not should_process_mask(mask_path):
        return
    orig_img_path = get_original_image_path(mask_path)
    if not orig_img_path:
        print(f"Original image not found for mask: {mask_path}")
        return
        
    img = cv2.imread(orig_img_path)
    if img is None:
        print(f"Failed to read image: {orig_img_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        print(f"Failed to read mask: {mask_path}")
        return
        
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    overlay = img.copy()
    
    for c in contours:
        # Draw green contour line
        cv2.drawContours(overlay, [c], -1, (0, 255, 0), thickness=2)
        # Draw red bounding box
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (255, 0, 0), thickness=2)
        
    if show:
        plt.figure(figsize=(15, 5))
        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.title('Original Image')
        plt.axis('off')
        
        plt.subplot(1, 3, 2)
        plt.imshow(mask, cmap='gray')
        plt.title('Mask')
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(overlay)
        plt.title('Overlayed Contours & BBox')
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
    if output_path:
        overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_dir = os.path.dirname(output_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
        cv2.imwrite(output_path, overlay_rgb)
        print(f"Saved to {output_path}")

### Test Overlays Inline

In [6]:
example_polyp = [p for p in glob.glob(os.path.join(DATA_DIR, "polyp_masks", "*.jpg")) if should_process_mask(p)]
example_instrument = [p for p in glob.glob(os.path.join(DATA_DIR, "instruments_masks", "*.jpg")) if should_process_mask(p)]
example_pseudo = [p for p in glob.glob(os.path.join(DATA_DIR, "pseudo_masks", "*.jpg")) if should_process_mask(p)]
example_gradcam = [p for p in glob.glob(os.path.join(DATA_DIR, "gradcam_masks", "*", "*", "mask.png")) if should_process_mask(p)]

print("Testing Polyp Mask")
if example_polyp: process_mask_and_image(example_polyp[20])

print("Testing Instrument Mask")
if example_instrument: process_mask_and_image(example_instrument[20])

print("Testing Pseudo Mask (Cecum/Z-line)")
if example_pseudo: process_mask_and_image(example_pseudo[20])

print("Testing Gradcam Mask")
if example_gradcam: process_mask_and_image(example_gradcam[20])

Testing Polyp Mask
Testing Instrument Mask
Testing Pseudo Mask (Cecum/Z-line)
Testing Gradcam Mask
